In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from wordfreq import zipf_frequency, top_n_list

# Funkcja obliczająca współczynnik pewności języka
Oblicza współczynnik pewności określający,
jak dobrze dany tekst pasuje do konkretnego języka.

Współczynnik liczony jest jako ważona suma częstotliwości słów,
które występują zarówno w analizowanym tekście,
jak i wśród k najczęściej występujących słów danego języka.
Wynik jest normalizowany przez łączną liczbę słów w tekście.

Wyższe wartości oznaczają lepsze dopasowanie językowe.

In [ ]:
def lang_confidence_score(word_counts, language_words_with_frequency, k):

    total_words = sum(word_counts.values())
    if total_words == 0:
        return 0.0

    score = 0.0

    for word, lang_freq in language_words_with_frequency[:k]:
        if word in word_counts:
            score += word_counts[word] * lang_freq

    return score / total_words

# Narzędzia do wczytywania danych

Wczytuje słownik zliczeń słów z pliku JSON.

In [ ]:
def load_word_counts(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

Zwraca listę par (słowo, częstotliwość) dla top-N
najczęściej występujących słów danego języka,
korzystając z częstotliwości Zipfa z biblioteki wordfreq.

In [ ]:
def get_top_language_words(language_code, top_n=1000):
    words = top_n_list(language_code, top_n)
    return [(word, zipf_frequency(word, language_code)) for word in words]

# Prezentacja wyników w formie tabelarycznej
Wypisuje osobne, czytelne tabele dla każdego analizowanego tekstu.
Wiersze odpowiadają różnym wartościom k,
kolumny odpowiadają językom.

In [ ]:
def print_readable_tables(results_df):

    pd.set_option("display.precision", 4)

    for text_name in results_df["text"].unique():
        print("\n" + "=" * 70)
        print(f"TEXT: {text_name}")
        print("=" * 70)

        table = (
            results_df[results_df["text"] == text_name]
            .pivot(index="k", columns="language", values="score")
            .sort_index()
            .round(4)
        )

        print(table)

# Wizualizacja wyników
Generuje wykresy liniowe pokazujące,
jak współczynnik pewności języka zależy od parametru k
dla każdego analizowanego tekstu.

In [ ]:
def plot_results(results_df):

    for text_name in results_df["text"].unique():
        subset = results_df[results_df["text"] == text_name]

        plt.figure(figsize=(7, 5))

        for language in subset["language"].unique():
            lang_data = subset[subset["language"] == language]
            plt.plot(
                lang_data["k"],
                lang_data["score"],
                marker="o",
                label=language
            )

        plt.title(f"Współczynniki pewności języka dla: {text_name}")
        plt.xlabel("k (liczba najczęstszych słów języka)")
        plt.ylabel("Współczynnik pewności języka")
        plt.xscale("log")
        plt.legend()
        plt.grid(True, which="both", linestyle="--", alpha=0.5)
        plt.tight_layout()
        plt.show()

# Główna analiza
Źródła danych:

Bad-word-counts.json
https://tolkiengateway.net/wiki/Cor_Blok

Big-word-counts.json
https://tolkiengateway.net/wiki/The_Lord_of_the_Rings:_The_Return_of_the_King

English-word-counts.json
"Pride and Prejudice" – Jane Austen (pierwsze ~4884 słowa)

Polish-word-counts.json
"Pan Tadeusz" – Adam Mickiewicz (pierwsze ~5144 słowa)

French-word-counts.json
"Les Misérables" – Victor Hugo (pierwsze ~5945 słów)

In [ ]:
if __name__ == "__main__":

    languages = {
        "English": "en",
        "Polish": "pl",
        "French": "fr",
    }

    k_values = [3, 10, 100, 1000]

    text_files = {
        "Big_Wiki_EN": "Big-word-counts.json",
        "Bad_Wiki_EN": "Bad-word-counts.json",
        "Text_EN": "English-word-counts.json",
        "Text_PL": "Polish-word-counts.json",
        "Text_FR": "French-word-counts.json",
    }

    word_counts_data = {
        name: load_word_counts(path)
        for name, path in text_files.items()
    }

    language_frequency_lists = {
        lang_name: get_top_language_words(lang_code, top_n=1000)
        for lang_name, lang_code in languages.items()
    }

    results = []

    for text_name, word_counts in word_counts_data.items():
        for lang_name, lang_words in language_frequency_lists.items():
            for k in k_values:
                score = lang_confidence_score(
                    word_counts,
                    lang_words,
                    k,
                )

                results.append({
                    "text": text_name,
                    "language": lang_name,
                    "k": k,
                    "score": score,
                })

    results_df = pd.DataFrame(results)

    print_readable_tables(results_df)
    plot_results(results_df)

# Empiryczna analiza wyników

1. Zachowanie funkcji w zależności od parametru k

Można zauważyć wyraźny wzrost wartości lang_confidence_score wraz ze zwiększeniem liczby najczestszych słów k.
Dla bardzo małych wartości, funkcja nie potrafi jednoznacznie rozróznić języka tekstu. 
Dopiero dla k >= 100 widoczna jest stabilizacja i wyraźna separacja wyników. 

2. Teksty z tolkien_gateway wiki

Jako że wiki jest napisane w języku angielskim, widać że wyniki są wyższe w tym języku dla Bad_Wiki_EN i Big_wiki_EN niż w pozostałych. 
Dla długiego artykułu funkcja działa bardzo skutecznie. Dla każdego k zawsze osiąga najwyższy wynik w języku angielskim. 
Dla specjalnie źle dobranego działa gorzej ale wciąż osiąga duży wynik. 

3. Teksty spoza wiki 

Funkcja tutaj równie skutecznie wykrywa język tesktu. 
W szczególności angielski i francuski tekst osiągają najwyższe wyniki dla swoich języków w całym programie.
Jedynie język polski osiąga dosyć niski wynik w porównaniu do reszty. Wynikać to może z tego że różne odmiany słów są przez program
widziane jako oddzielne słowa, oraz sposób napisania "Pana Tadeusza" (trzynastozgłoskowcem) wymusza dobór nie tak popularnych słów. 

4. Czy dobór języków miał duże znaczenie?

Dobór języków miał zauważalne znaczenie. Języki należące do różnych rodzin (angielski – germański, francuski – romański, polski – słowiański) 
są relatywnie łatwe do odróżnienia, szczególnie dla większych k.

5. Czy po wartościach language_words_with_frequency dla danych i najczęstszych 
słowach z języka danych widać, że w wybranym języku słowa często są odmieniane?

Analiza wyników sugeruje, że w językach silnie fleksyjnych (np. tak jak wcześniej zauważono polski)
najczęstsze słowa nie pokrywają tak dużej części tekstu jak w językach o mniejszej fleksyjności. 

6. Czy trudne było znalezienie takiego artykułu, dla którego wynik
language_words_with_frequency jest jak najmniejszy w języku wiki? Czy to specyfika tego wiki?

Znalezienie tego konkretnego artykułu wymagało przeanalizowania licznych źródeł. Wiki to jest redagowane
w przystępnym, współczesnym języku angielskim, przez co większość zawartego w niej słownictwa to powszechnie używane wyrazy. 
Nie jest to jednak specyfika tylko tego wiki.
Przeważająca część różnych wiki charakteryzuje się podobnym stylem.
W związku z tym, wyniki analizy dla naszego zbioru language_words_with_frequency w przypadku poszczególnych artykułów 
zależałyby przede wszystkim od ich stopnia specjalizacji.